# Pitwall: fine-tune the router model (free Colab GPU)

Trains a LoRA on Qwen3-0.6B-Base to pick Gearbox's tier (S/M/L) for a prompt, then
exports a quantised GGUF bundle (~400 MB) to download. About 20 minutes.

1. **Runtime -> Change runtime type -> T4 GPU**
2. **Runtime -> Run all**
3. Save `ft-train_old.zip` into `bundles/` of your Pitwall checkout and unzip it there.

The notebook only calls the repo's training script (`pitwall.finetune.train`); its
docstring explains each step and why. A failing `!` command does not stop *Run all*, so
every step checks `_exit_code` and stops the notebook on a failure.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip uninstall -y torchao

In [ ]:
!git pull

In [ ]:
# The Pitwall commit to train from. "main" = latest; a commit id makes the run repeatable.
COMMIT = "main"
!git clone -q https://github.com/Utsav0706/pitwall.git
%cd pitwall
!git checkout -q {COMMIT} && git log -1 --format='training from %h %s'
assert _exit_code == 0, "checkout failed"

In [ ]:
# Colab's own torch is kept (it matches the GPU driver); everything else is pinned.
!pip install -q -e . -r training/requirements-colab.txt
assert _exit_code == 0, "install failed"
# Colab ships torchao 0.10, which peft refuses to load beside (it needs >= 0.16).
# Pitwall doesn't use torchao, so remove it rather than upgrade it against Colab's torch.
!pip uninstall -q -y torchao

In [ ]:
# Rebuild the training data exactly: fetch Gearbox's files from the pinned commit, build
# the pairs, and stop if anything differs from the committed dvc.lock.
!dvc update -q -R data/gearbox
assert _exit_code == 0, "dvc update failed"
!dvc repro -q
assert _exit_code == 0, "dvc repro failed"
!git diff --exit-code --stat
assert _exit_code == 0, "data differs from dvc.lock - stop"
!pitwall data show

In [13]:
!python -m pitwall.finetune.train     --train data/derived/train_old.jsonl     --out bundles/ft-train_old     --llama-cpp .llama.cpp
assert _exit_code == 0, "training failed - see the output above"

device cuda; 292 train, 73 validation
Loading weights: 100% 310/310 [00:00<00:00, 355.18it/s]
/content/pitwall/pitwall/src/pitwall/finetune/train.py:225: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  schedule.step()
/content/pitwall/pitwall/src/pitwall/finetune/train.py:226: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(chunk)
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<fr

AssertionError: training failed - see the output above

In [ ]:
!pitwall bundle check bundles/ft-train_old
assert _exit_code == 0, "bundle check failed"
import shutil

archive = shutil.make_archive("ft-train_old", "zip", "bundles", "ft-train_old")
print(archive)

In [ ]:
# Download. If the browser download stalls (~400 MB), use the Drive cell below instead.
from google.colab import files

files.download("ft-train_old.zip")

In [ ]:
# Optional: copy to Google Drive instead (asks for Drive access).
# from google.colab import drive
# drive.mount("/content/drive")
# !cp ft-train_old.zip /content/drive/MyDrive/